To Do: Test the two architectures with 50 epochs and same training parameters as before and on the same train-test split.
Also save the confusion matrix image for these two runs.

1. ResNet50 MultiLevel MultiScale Architecture

In [ ]:
def inception_block(inputs, filters, name=None):
    """
    GoogLeNet-style Inception block.

    Four branches:

        Branch 1:
            1x1

        Branch 2:
            1x1 -> 3x3

        Branch 3:
            1x1 -> 5x5

        Branch 4:
            3x3 MaxPool -> 1x1

    The output channel count is equal to `filters`.
    Spatial dimensions are preserved.
    """

    # --------------------------------------------------------
    # Branch 1
    # 1x1
    # --------------------------------------------------------

    branch1 = tf.keras.layers.Conv2D(
        filters=filters["branch1"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch1_1x1"
    )(inputs)


    # --------------------------------------------------------
    # Branch 2
    # 1x1 -> 3x3
    # --------------------------------------------------------

    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_reduce"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch2_reduce"
    )(inputs)

    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_out"],
        kernel_size=(3, 3),
        padding="same",
        activation="relu",
        name=f"{name}_branch2_3x3"
    )(branch2)


    # --------------------------------------------------------
    # Branch 3
    # 1x1 -> 5x5
    # --------------------------------------------------------

    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_reduce"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch3_reduce"
    )(inputs)

    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_out"],
        kernel_size=(5, 5),
        padding="same",
        activation="relu",
        name=f"{name}_branch3_5x5"
    )(branch3)


    # --------------------------------------------------------
    # Branch 4
    # MaxPool -> 1x1
    # --------------------------------------------------------

    branch4 = tf.keras.layers.MaxPooling2D(
        pool_size=(3, 3),
        strides=(1, 1),
        padding="same",
        name=f"{name}_branch4_pool"
    )(inputs)

    branch4 = tf.keras.layers.Conv2D(
        filters=filters["branch4_out"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch4_1x1"
    )(branch4)


    # --------------------------------------------------------
    # Concatenate
    # --------------------------------------------------------

    outputs = tf.keras.layers.Concatenate(
        axis=-1,
        name=f"{name}_concat"
    )([
        branch1,
        branch2,
        branch3,
        branch4
    ])

    return outputs




inputs = tf.keras.Input(shape=(224, 224, 3))

x = tf.keras.applications.resnet50.preprocess_input(inputs)

base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_tensor=x
)

# Fine-tuning enabled
base_model.trainable = True



# -------------------------
# Extract intermediate layers
# -------------------------
l3 = base_model.get_layer("conv3_block4_add").output   # 28x28x512
l4 = base_model.get_layer("conv4_block6_add").output   # 14x14x1024
l5 = base_model.get_layer("conv5_block3_add").output   # 7x7x2048


# ============================================================
# INCEPTION FILTER CONFIGURATIONS
# ============================================================

l3_filters = {
    "branch1": 64,

    "branch2_reduce": 96,
    "branch2_out": 104,

    "branch3_reduce": 32,
    "branch3_out": 172,

    "branch4_out": 172
}


l4_filters = {
    "branch1": 128,

    "branch2_reduce": 192,
    "branch2_out": 208,

    "branch3_reduce": 64,
    "branch3_out": 344,

    "branch4_out": 344
}


l5_filters = {
    "branch1": 256,

    "branch2_reduce": 384,
    "branch2_out": 416,

    "branch3_reduce": 128,
    "branch3_out": 688,

    "branch4_out": 688
}


# ============================================================
# APPLY INCEPTION BLOCKS
# ============================================================

i3 = inception_block(
    l3,
    filters=l3_filters,
    name="inception_l3"
)
# i3: 28 x 28 x 512


i4 = inception_block(
    l4,
    filters=l4_filters,
    name="inception_l4"
)
# i4: 14 x 14 x 1024


i5 = inception_block(
    l5,
    filters=l5_filters,
    name="inception_l5"
)
# i5: 7 x 7 x 2048



x3 = tf.keras.layers.LeakyReLU(
    negative_slope=0.1
)(i3)

x3 = tf.keras.layers.MaxPooling2D(
    pool_size=(2, 2)
)(x3)


# ============================================================
# L4
# 14 x 14 x 1024
# ============================================================

x4 = i4


# ============================================================
# L5
# 7 x 7 x 2048 -> 14 x 14 x 2048
# ============================================================

x5 = tf.keras.layers.LeakyReLU(
    negative_slope=0.1
)(i5)

x5 = tf.keras.layers.Conv2DTranspose(
    filters=2048,
    kernel_size=(2, 2),
    strides=(2, 2),
    padding="same"
)(x5)

x5 = tf.keras.layers.BatchNormalization()(x5)

x5 = tf.keras.layers.LeakyReLU(
    negative_slope=0.1
)(x5)


# ============================================================
# FUSION
# ============================================================

fusion = tf.keras.layers.Concatenate(
    axis=-1
)([x3, x4, x5])


# # -------------------------
# # Classification Head
# # -------------------------
x = tf.keras.layers.GlobalAveragePooling2D()(fusion)

x = tf.keras.layers.BatchNormalization()(x)

x = tf.keras.layers.Dense(2048, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)

x = tf.keras.layers.Dense(1024, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)




2. ResNet50 Architecture MultiLevel Attention

In [ ]:
def se_block(inputs, reduction=16, name=None):
    """
    Standard Squeeze-and-Excitation Block
    """

    channels = inputs.shape[-1]

    x = tf.keras.layers.GlobalAveragePooling2D(
        name=f"{name}_gap" if name else None
    )(inputs)

    x = tf.keras.layers.Dense(
        channels // reduction,
        activation="relu",
        use_bias=True,
        name=f"{name}_fc1" if name else None
    )(x)

    x = tf.keras.layers.Dense(
        channels,
        activation="sigmoid",
        use_bias=True,
        name=f"{name}_fc2" if name else None
    )(x)

    x = tf.keras.layers.Reshape(
        (1, 1, channels),
        name=f"{name}_reshape" if name else None
    )(x)

    outputs = tf.keras.layers.Multiply(
        name=f"{name}_scale" if name else None
    )([inputs, x])

    return outputs


inputs = tf.keras.Input(shape=(224, 224, 3))

x = tf.keras.applications.resnet50.preprocess_input(inputs)

base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_tensor=x
)

# Fine-tuning enabled
base_model.trainable = True



# -------------------------
# Extract intermediate layers
# -------------------------
l3 = base_model.get_layer("conv3_block4_add").output   # 28x28x512
l4 = base_model.get_layer("conv4_block6_add").output   # 14x14x1024
l5 = base_model.get_layer("conv5_block3_add").output   # 7x7x2048


l3_se = se_block(l3, reduction=16, name="se3")   # 28x28x512
l4_se = se_block(l4, reduction=16, name="se4")   # 14x14x1024
l5_se = se_block(l5, reduction=16, name="se5")   # 7x7x2048


# # -------------------------
# # 1. conv3_block4_add + SE → 14x14x512
# # -------------------------
 x3 = tf.keras.layers.LeakyReLU(alpha=0.1)(l3_se)
 x3 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x3)


 # -------------------------
 # 2. conv4_block6_add + SE → unchanged
 # -------------------------
 x4 = l4_se


 # -------------------------
 # 3. conv5_block3_add + SE → 14x14x2048
 # -------------------------
 x5 = tf.keras.layers.LeakyReLU(alpha=0.1)(l5_se)

 x5 = tf.keras.layers.Conv2DTranspose(
     filters=2048,
     kernel_size=(2, 2),
     strides=(2, 2),
     padding="same"
)(x5)

x5 = tf.keras.layers.BatchNormalization()(x5)
x5 = tf.keras.layers.LeakyReLU(alpha=0.1)(x5)


# # -------------------------
# # Feature Fusion
# # -------------------------
fusion = tf.keras.layers.Concatenate(axis=-1)([x3, x4, x5])
# # Shape: (None, 14, 14, 3584)


# # -------------------------
# # Classification Head
# # -------------------------
x = tf.keras.layers.GlobalAveragePooling2D()(fusion)

x = tf.keras.layers.BatchNormalization()(x)

x = tf.keras.layers.Dense(2048, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)

x = tf.keras.layers.Dense(1024, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

Architecture 3

In [ ]:
def se_block(inputs, reduction, name=None):
    """
    Standard Squeeze-and-Excitation Block
    """

    channels = inputs.shape[-1]

    x = tf.keras.layers.GlobalAveragePooling2D(
        name=f"{name}_gap" if name else None
    )(inputs)

    x = tf.keras.layers.Dense(
        channels // reduction,
        activation="relu",
        use_bias=True,
        name=f"{name}_fc1" if name else None
    )(x)

    x = tf.keras.layers.Dense(
        channels,
        activation="sigmoid",
        use_bias=True,
        name=f"{name}_fc2" if name else None
    )(x)

    x = tf.keras.layers.Reshape(
        (1, 1, channels),
        name=f"{name}_reshape" if name else None
    )(x)

    outputs = tf.keras.layers.Multiply(
        name=f"{name}_scale" if name else None
    )([inputs, x])

    return outputs



def inception_attention_block(inputs, filters, name=None):

    # ==========================================
    # Branch 1: 1x1
    # ==========================================
    branch1 = tf.keras.layers.Conv2D(
        filters=filters["branch1"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch1_1x1"
    )(inputs)

    branch1 = se_block(
        branch1,
        reduction=16,
        name=f"{name}_branch1_se"
    )


    # ==========================================
    # Branch 2: 1x1 -> 3x3
    # ==========================================
    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_reduce"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch2_reduce"
    )(inputs)

    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_out"],
        kernel_size=(3, 3),
        padding="same",
        activation="relu",
        name=f"{name}_branch2_3x3"
    )(branch2)

    branch2 = se_block(
        branch2,
        reduction=16,
        name=f"{name}_branch2_se"
    )


    # ==========================================
    # Branch 3: 1x1 -> 5x5
    # ==========================================
    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_reduce"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch3_reduce"
    )(inputs)

    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_out"],
        kernel_size=(5, 5),
        padding="same",
        activation="relu",
        name=f"{name}_branch3_5x5"
    )(branch3)

    branch3 = se_block(
        branch3,
        reduction=16,
        name=f"{name}_branch3_se"
    )


    # ==========================================
    # Branch 4: MaxPool -> 1x1
    # ==========================================
    branch4 = tf.keras.layers.MaxPooling2D(
        pool_size=(3, 3),
        strides=(1, 1),
        padding="same",
        name=f"{name}_branch4_pool"
    )(inputs)

    branch4 = tf.keras.layers.Conv2D(
        filters=filters["branch4_out"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch4_1x1"
    )(branch4)

    branch4 = se_block(
        branch4,
        reduction=16,
        name=f"{name}_branch4_se"
    )


    # ==========================================
    # Concatenate
    # ==========================================
    outputs = tf.keras.layers.Concatenate(
        axis=-1,
        name=f"{name}_concat"
    )([
        branch1,
        branch2,
        branch3,
        branch4
    ])

    return outputs


inputs = tf.keras.Input(shape=(224, 224, 3))

x = tf.keras.applications.resnet50.preprocess_input(inputs)

base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_tensor=x
)

# Fine-tuning enabled
base_model.trainable = True



# -------------------------
# Extract intermediate layers
# -------------------------
l3 = base_model.get_layer("conv3_block4_add").output   # 28x28x512
l4 = base_model.get_layer("conv4_block6_add").output   # 14x14x1024
l5 = base_model.get_layer("conv5_block3_add").output   # 7x7x2048


# ============================================================
# INCEPTION FILTER CONFIGURATIONS
# ============================================================

l3_filters = {
    "branch1": 64,

    "branch2_reduce": 96,
    "branch2_out": 104,

    "branch3_reduce": 32,
    "branch3_out": 172,

    "branch4_out": 172
}


l4_filters = {
    "branch1": 128,

    "branch2_reduce": 192,
    "branch2_out": 208,

    "branch3_reduce": 64,
    "branch3_out": 344,

    "branch4_out": 344
}


l5_filters = {
    "branch1": 256,

    "branch2_reduce": 384,
    "branch2_out": 416,

    "branch3_reduce": 128,
    "branch3_out": 688,

    "branch4_out": 688
}



i3 = inception_attention_block(
    l3,
    filters=l3_filters,
    name="inception_l3"
)

i4 = inception_attention_block(
    l4,
    filters=l4_filters,
    name="inception_l4"
)

i5 = inception_attention_block(
    l5,
    filters=l5_filters,
    name="inception_l5"
)



x3 = tf.keras.layers.LeakyReLU(
    negative_slope=0.1
)(i3)

x3 = tf.keras.layers.MaxPooling2D(
    pool_size=(2, 2)
)(x3)


# ============================================================
# L4
# 14 x 14 x 1024
# ============================================================

x4 = i4


# ============================================================
# L5
# 7 x 7 x 2048 -> 14 x 14 x 2048
# ============================================================

x5 = tf.keras.layers.LeakyReLU(
    negative_slope=0.1
)(i5)

x5 = tf.keras.layers.Conv2DTranspose(
    filters=2048,
    kernel_size=(2, 2),
    strides=(2, 2),
    padding="same"
)(x5)

x5 = tf.keras.layers.BatchNormalization()(x5)

x5 = tf.keras.layers.LeakyReLU(
    negative_slope=0.1
)(x5)


# ============================================================
# FUSION
# ============================================================

fusion = tf.keras.layers.Concatenate(
    axis=-1
)([x3, x4, x5])


# # -------------------------
# # Classification Head
# # -------------------------
x = tf.keras.layers.GlobalAveragePooling2D()(fusion)

x = tf.keras.layers.BatchNormalization()(x)

x = tf.keras.layers.Dense(2048, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)

x = tf.keras.layers.Dense(1024, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)



Architecture 4

In [ ]:
def inception_block(inputs, filters, name=None):
    """
    GoogLeNet-style Inception block.

    Four branches:

        Branch 1:
            1x1

        Branch 2:
            1x1 -> 3x3

        Branch 3:
            1x1 -> 5x5

        Branch 4:
            3x3 MaxPool -> 1x1

    The output channel count is equal to `filters`.
    Spatial dimensions are preserved.
    """

    # --------------------------------------------------------
    # Branch 1
    # 1x1
    # --------------------------------------------------------

    branch1 = tf.keras.layers.Conv2D(
        filters=filters["branch1"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch1_1x1"
    )(inputs)


    # --------------------------------------------------------
    # Branch 2
    # 1x1 -> 3x3
    # --------------------------------------------------------

    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_reduce"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch2_reduce"
    )(inputs)

    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_out"],
        kernel_size=(3, 3),
        padding="same",
        activation="relu",
        name=f"{name}_branch2_3x3"
    )(branch2)


    # --------------------------------------------------------
    # Branch 3
    # 1x1 -> 5x5
    # --------------------------------------------------------

    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_reduce"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch3_reduce"
    )(inputs)

    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_out"],
        kernel_size=(5, 5),
        padding="same",
        activation="relu",
        name=f"{name}_branch3_5x5"
    )(branch3)


    # --------------------------------------------------------
    # Branch 4
    # MaxPool -> 1x1
    # --------------------------------------------------------

    branch4 = tf.keras.layers.MaxPooling2D(
        pool_size=(3, 3),
        strides=(1, 1),
        padding="same",
        name=f"{name}_branch4_pool"
    )(inputs)

    branch4 = tf.keras.layers.Conv2D(
        filters=filters["branch4_out"],
        kernel_size=(1, 1),
        padding="same",
        activation="relu",
        name=f"{name}_branch4_1x1"
    )(branch4)


    # --------------------------------------------------------
    # Concatenate
    # --------------------------------------------------------

    outputs = tf.keras.layers.Concatenate(
        axis=-1,
        name=f"{name}_concat"
    )([
        branch1,
        branch2,
        branch3,
        branch4
    ])

    return outputs




inputs = tf.keras.Input(shape=(224, 224, 3))

x = tf.keras.applications.resnet50.preprocess_input(inputs)

base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_tensor=x
)

# Fine-tuning enabled
base_model.trainable = True



# -------------------------
# Extract intermediate layers
# -------------------------
l3 = base_model.get_layer("conv3_block4_add").output   # 28x28x512
l4 = base_model.get_layer("conv4_block6_add").output   # 14x14x1024
l5 = base_model.get_layer("conv5_block3_add").output   # 7x7x2048


# ============================================================
# INCEPTION FILTER CONFIGURATIONS
# ============================================================

l3_filters = {
    "branch1": 64,

    "branch2_reduce": 96,
    "branch2_out": 104,

    "branch3_reduce": 32,
    "branch3_out": 172,

    "branch4_out": 172
}


l4_filters = {
    "branch1": 128,

    "branch2_reduce": 192,
    "branch2_out": 208,

    "branch3_reduce": 64,
    "branch3_out": 344,

    "branch4_out": 344
}


l5_filters = {
    "branch1": 256,

    "branch2_reduce": 384,
    "branch2_out": 416,

    "branch3_reduce": 128,
    "branch3_out": 688,

    "branch4_out": 688
}


# ============================================================
# APPLY INCEPTION BLOCKS
# ============================================================

i3 = inception_block(
    l3,
    filters=l3_filters,
    name="inception_l3"
)
# i3: 28 x 28 x 512


i4 = inception_block(
    l4,
    filters=l4_filters,
    name="inception_l4"
)
# i4: 14 x 14 x 1024


i5 = inception_block(
    l5,
    filters=l5_filters,
    name="inception_l5"
)
# i5: 7 x 7 x 2048



x3 = tf.keras.layers.LeakyReLU(
    negative_slope=0.1
)(i3)

x3 = tf.keras.layers.MaxPooling2D(
    pool_size=(2, 2)
)(x3)


# ============================================================
# L4
# 14 x 14 x 1024
# ============================================================

x4 = i4


# ============================================================
# L5
# 7 x 7 x 2048 -> 14 x 14 x 2048
# ============================================================

x5 = tf.keras.layers.LeakyReLU(
    negative_slope=0.1
)(i5)

# 7×7×2048 → 14×14×2048
x5 = tf.keras.layers.UpSampling2D(
    size=(2, 2),
    interpolation="bicubic"
)(x5)

# Spatial feature refinement
x5 = tf.keras.layers.DepthwiseConv2D(
    kernel_size=(3, 3),
    padding="same",
    use_bias=False
)(x5)

x5 = tf.keras.layers.BatchNormalization()(x5)
x5 = tf.keras.layers.LeakyReLU(
    negative_slope=0.1
)(x5)

# Channel mixing
x5 = tf.keras.layers.Conv2D(
    filters=2048,
    kernel_size=(1, 1),
    padding="same",
    use_bias=False
)(x5)

x5 = tf.keras.layers.BatchNormalization()(x5)
x5 = tf.keras.layers.LeakyReLU(
    negative_slope=0.1
)(x5)

# ============================================================
# FUSION
# ============================================================

fusion = tf.keras.layers.Concatenate(
    axis=-1
)([x3, x4, x5])


# # -------------------------
# # Classification Head
# # -------------------------
x = tf.keras.layers.GlobalAveragePooling2D()(fusion)

x = tf.keras.layers.BatchNormalization()(x)

x = tf.keras.layers.Dense(2048, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)

x = tf.keras.layers.Dense(1024, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)


